# Examen ICCD753 Recuperación de Información

## Estudiante : Eddy Alejandro Sangucho Coronel

## Corpus asignado : "SciFact"

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Objetivo
Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.


## 1. Instalación y configuración


In [2]:
%pip install -q datasets scikit-learn rank-bm25 pandas matplotlib

In [3]:
import re, math, json
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"(?u)\b\w+\b", str(text).lower())

## 2. Descarga y exploración de SciFact
Se cargan el corpus, las consultas y los qrels publicados por BeIR. Luego se imprimen splits y columnas para auditar los datos.

In [4]:
def cargar_datasets_scifact():
    """Carga corpus, consultas y qrels oficiales."""
    return (load_dataset("BeIR/scifact", "corpus"),
            load_dataset("BeIR/scifact", "queries"),
            load_dataset("BeIR/scifact-qrels"))

corpus_ds, queries_ds, qrels_ds = cargar_datasets_scifact()
print("Corpus:", corpus_ds)
print("Consultas:", queries_ds)
print("Qrels:", qrels_ds)

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

corpus/corpus-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.47MB            

corpus/corpus-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating corpus split:   0%|          | 0/5183 [00:00<?, ? examples/s]

queries/queries-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 65.0kB            

queries/queries-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (3 outputs left out)

test.tsv:   0%|          | 0.00/5.39k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/919 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/339 [00:00<?, ? examples/s]

Corpus: DatasetDict({
    corpus: Dataset({
        features: ['_id', 'title', 'text'],
        num_rows: 5183
    })
})
Consultas: DatasetDict({
    queries: Dataset({
        features: ['_id', 'title', 'text'],
        num_rows: 1109
    })
})
Qrels: DatasetDict({
    train: Dataset({
        features: ['query-id', 'corpus-id', 'score'],
        num_rows: 919
    })
    test: Dataset({
        features: ['query-id', 'corpus-id', 'score'],
        num_rows: 339
    })
})


In [5]:
def convertir_splits(dataset):
    return {split: [dict(row) for row in dataset[split]] for split in dataset}

def inspeccionar_datasets(corpus_ds, queries_ds, qrels_ds):
    datos = {"corpus": convertir_splits(corpus_ds),
             "consultas": convertir_splits(queries_ds),
             "qrels": convertir_splits(qrels_ds)}
    for nombre, splits in datos.items():
        print(nombre, {split: len(rows) for split, rows in splits.items()})
    return datos

datos = inspeccionar_datasets(corpus_ds, queries_ds, qrels_ds)
corpus_rows, query_rows, qrel_rows = datos["corpus"], datos["consultas"], datos["qrels"]

corpus {'corpus': 5183}
consultas {'queries': 1109}
qrels {'train': 919, 'test': 339}


## 3. Preparación e índice indexación
* Cada fila representa un documento original.
* Se concatenan título y texto y no se crean chunks.
* TF-IDF usa ponderación sublinear de frecuencia y normalización L2, por lo que el producto escalar con la consulta equivale a coseno.
* BM25 Okapi usa k1=1.5 y b=0.75.
* El vectorizador expone el vocabulario y la matriz dispersa es el índice invertido computable.

In [6]:
def construir_indice(corpus_splits, tokenizer):
    """Prepara documentos completos e índices TF-IDF y BM25."""
    filas = corpus_splits.get("corpus", next(iter(corpus_splits.values())))
    ids = [str(d.get("_id", d.get("id"))) for d in filas]
    titulos = [d.get("title", "") for d in filas]
    textos = [(d.get("title", "") + " " + d.get("text", "")).strip() for d in filas]
    if len(ids) != len(set(ids)):
        raise ValueError("Hay IDs duplicados en el corpus")
    vectorizador = TfidfVectorizer(tokenizer=tokenizer, token_pattern=None, sublinear_tf=True)
    matriz = vectorizador.fit_transform(textos)
    bm25_model = BM25Okapi([tokenizer(t) for t in textos], k1=1.5, b=0.75)
    return {"ids": ids, "titulos": titulos, "textos": textos,
            "vectorizador": vectorizador, "matriz": matriz, "bm25": bm25_model}

indice = construir_indice(corpus_rows, tokenize)
doc_ids, titles, doc_texts = indice["ids"], indice["titulos"], indice["textos"]
vectorizer, tfidf_matrix, bm25 = indice["vectorizador"], indice["matriz"], indice["bm25"]
print(f"Documentos: {len(doc_ids):,}; vocabulario: {len(vectorizer.vocabulary_):,}; matriz: {tfidf_matrix.shape}")

Documentos: 5,183; vocabulario: 35,954; matriz: (5183, 35954)


In [7]:
def retrieve(query, model="bm25", top_k=10, index=indice):
    """Retorna documentos únicos ordenados por puntuación descendente."""
    if model not in {"bm25", "tfidf"}: raise ValueError("modelo inválido")
    if not 1 <= int(top_k) <= 100: raise ValueError("top_k debe ser 1..100")
    if model == "tfidf":
        q = index["vectorizador"].transform([query])
        scores = (index["matriz"] @ q.T).toarray().ravel()
    else: scores = np.asarray(index["bm25"].get_scores(tokenize(query)))
    order = sorted(range(len(scores)), key=lambda i: (-float(scores[i]), index["ids"][i]))
    order = [i for i in order if scores[i] > 0][:int(top_k)]
    return [{"rank":r,"document_id":index["ids"][i],"score":float(scores[i]),
             "title":index["titulos"][i],"snippet":index["textos"][i][:500]}
            for r,i in enumerate(order,1)]

def comparar_modelos(query, top_k=5):
    return {m: retrieve(query,m,top_k) for m in ("tfidf","bm25")}

for model, ranking in comparar_modelos("Does vitamin D reduce cancer risk?", 2).items():
    print(model, ranking)

tfidf [{'rank': 1, 'document_id': '38551172', 'score': 0.29850713486294755, 'title': 'Mammographic density, plasma vitamin D levels and risk of breast cancer in postmenopausal women.', 'snippet': "Mammographic density, plasma vitamin D levels and risk of breast cancer in postmenopausal women. Mammographic density is a strong risk factor for breast cancer, but the underlying biology for this association is unknown. Studies suggest that vitamin D may reduce breast cancer risk and dietary vitamin D intake has been associated with reduced breast density. We conducted a case-control study nested within the Nurses' Health Study cohort consisting of 463 and 497 postmenopausal cases and controls,"}, {'rank': 2, 'document_id': '22843838', 'score': 0.2920896078992168, 'title': '[Vitamin D and latitude as environmental factors in multiple sclerosis].', 'snippet': '[Vitamin D and latitude as environmental factors in multiple sclerosis]. Multiple sclerosis (MS) shows a multifold increase in prevale

## 4. Evaluación en consultas test
Se emparejan consultas y qrels mediante su ID original. Precision@10 = relevantes recuperados / 10; Recall@10 = relevantes recuperados / total de relevantes juzgados; NDCG@10 usa ganancias graduadas `2^relevancia-1` y descuento logarítmico. Se promedian consultas con al menos un juicio positivo. El bloque reporta qué consultas fueron evaluadas.

In [8]:
def preparar_consultas_test(query_splits, qrel_splits):
    """Normaliza IDs y empareja consultas con qrels de test."""
    filas_q = query_splits.get("test", next(iter(query_splits.values())))
    queries = {str(r.get("_id", r.get("id"))): str(r.get("text", r.get("query", ""))) for r in filas_q}
    if "test" not in qrel_splits: raise ValueError("No hay qrels test")
    qrels = {}
    for r in qrel_splits["test"]:
        qid, did = str(r["query-id"]), str(r["corpus-id"])
        qrels.setdefault(qid,{})[did] = int(r["score"])
    common = sorted(set(queries) & set(qrels))
    if not common: raise ValueError("No coinciden IDs de consulta")
    return queries, qrels, common

test_queries, test_qrels, common = preparar_consultas_test(query_rows, qrel_rows)
print("Consultas test evaluables:", len(common))

Consultas test evaluables: 300


In [9]:
def calcular_metricas(query, qrels, model, k=10):
    ranking = retrieve(query, model, k)
    rel = [qrels.get(x["document_id"],0) for x in ranking]
    hits, total = sum(x>0 for x in rel), sum(x>0 for x in qrels.values())
    dcg = sum((2**v-1)/math.log2(i+2) for i,v in enumerate(rel))
    ideal = sorted((v for v in qrels.values() if v>0), reverse=True)[:k]
    idcg = sum((2**v-1)/math.log2(i+2) for i,v in enumerate(ideal))
    return {f"Precision@{k}":hits/k, f"Recall@{k}":hits/total if total else 0,
            f"NDCG@{k}":dcg/idcg if idcg else 0}

def evaluar_modelos(queries, qrels, query_ids, models=("tfidf","bm25"), k=10):
    filas = [{"query_id":qid,"model":m,**calcular_metricas(queries[qid],qrels[qid],m,k)}
             for qid in query_ids for m in models]
    detalle = pd.DataFrame(filas)
    promedio = detalle.groupby("model")[[f"Precision@{k}",f"Recall@{k}",f"NDCG@{k}"]].mean().sort_index()
    return detalle, promedio

per_query, metrics = evaluar_modelos(test_queries,test_qrels,common)
display(metrics)
display(per_query.head(10))

,Precision@10,Recall@10,NDCG@10
model,,,
bm25,0.085,0.7740,0.651893
tfidf,0.086,0.7785,0.632695


,query_id,model,Precision@10,Recall@10,NDCG@10
0,1,tfidf,0.0,0.0,0.00000
1,1,bm25,0.0,0.0,0.00000
2,100,tfidf,0.1,1.0,1.00000
3,100,bm25,0.1,1.0,1.00000
4,1012,tfidf,0.1,1.0,1.00000
5,1012,bm25,0.1,1.0,1.00000
6,1014,tfidf,0.1,1.0,1.00000
7,1014,bm25,0.1,1.0,1.00000
8,1019,tfidf,0.1,1.0,1.00000
9,1019,bm25,0.1,1.0,0.63093


## 5. Comparación e interpretación
La tabla anterior contiene los promedios macro.
Para comparar, se observa qué modelo obtiene mayor NDCG@10 (calidad del orden), junto con Precision@10 y Recall@10.

En general BM25 incorpora saturación de frecuencia y longitud documental; TF-IDF coseno normaliza los vectores y puede comportarse mejor ante consultas cortas.

In [10]:
def interpretar_metricas(metrics):
    for columna in metrics.columns:
        ganador = metrics[columna].idxmax()
        print(f"{columna}: mejor {ganador} ({metrics.loc[ganador,columna]:.4f})")
    print("Comparación basada en promedio macro de las consultas con qrels test.")

interpretar_metricas(metrics)

Precision@10: mejor tfidf (0.0860)
Recall@10: mejor tfidf (0.7785)
NDCG@10: mejor bm25 (0.6519)
Comparación basada en promedio macro de las consultas con qrels test.


## 6. Prueba del servicio web
* El servicio SciFact está desplegado en Vercel: https://examen-gamma-gray.vercel.app.

* El código fuente y el contrato OpenAPI están en el repositorio AlejandroSC28/Examen (main.py, requirements.txt, openapi.yaml). La interfaz interactiva de FastAPI se abre en https://examen-gamma-gray.vercel.app/docs.
Endpoints




### Probar

* Abre https://examen-gamma-gray.vercel.app/docs.
* Expande POST /search y pulsa Try it out.
* Escribe este cuerpo y pulsa Execute:

In [ ]:
#{
#  "query": "Does vitamin D reduce cancer risk?",
# "model": "bm25",
#  "top_k": 10
# }


* Comprueba que Server response muestra HTTP 200 y que Response body contiene results.
* Repetir la prueba cambiando model a tfidf.

## 7. Guardar resultados
La siguiente celda exporta métricas y evaluaciones por consulta. Mantén los outputs de este notebook guardados al entregar.

In [11]:
def guardar_metricas(metrics, per_query):
    metrics.to_csv("metricas_scifact.csv")
    per_query.to_csv("metricas_por_consulta_scifact.csv", index=False)
    return "metricas_scifact.csv", "metricas_por_consulta_scifact.csv"

print("Archivos guardados:", guardar_metricas(metrics, per_query))

Archivos guardados: ('metricas_scifact.csv', 'metricas_por_consulta_scifact.csv')
